# Brain2Qwerty — Combined Pipeline (V1+V3 preprocessing, V3 architecture) on Colab

End-to-end sentence decoding from MEG with the V3 hybrid **Mamba-2/attention** encoder,
the **combined V1+V3 preprocessing** (`CombinedBCBLPreprocessing`: V1 cleaning + typed-string
metadata + V3 CTC targets), the V1 TF-IDF paraphrase-cluster split, and optional
**subject subset selection**.

Pipeline: `CTC (char) → word-contrastive (SigLIP) → LoRA LLM decoder`, staged schedule.

**Before you start:**
1. Runtime → Change runtime type → **GPU** (T4 or better; A100 recommended for the full encoder).
2. Request access to the gated dataset [bcbl190626/SpanishBCBL](https://huggingface.co/datasets/bcbl190626/SpanishBCBL) and have your HF token ready.
3. Upload this repo (or just the `brain2qwerty_colab/` folder) to Google Drive, or clone your fork in the next cells.

In [ ]:
# 1. Check the GPU
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2. Install dependencies (torch/torchaudio are preinstalled on Colab)
!pip install -q neuralset==0.2.2 neuraltrain==0.2.2 neuralfetch==0.2.2 exca==0.5.22 submitit==1.5.3 \
    lightning==2.5.2 torchmetrics==1.7.3 x-transformers==2.4.9 transformers==4.52.4 peft==0.18.1 \
    numpy==2.2.6 pandas==2.2.3 scipy==1.14.1 scikit-learn==1.8.0 pydantic==2.12.5 \
    mne==1.11.0 dtw-python==1.7.4 edit_distance==1.0.7 Levenshtein==0.27.1 g2p_en==2.1.0 \
    regex tqdm pyyaml huggingface_hub

In [ ]:
# 3. Get the code onto the VM
# Option A: mount Drive and point at an uploaded copy of the project folder
from google.colab import drive
drive.mount('/content/drive')

import sys, os
# Path to the directory that CONTAINS the brain2qwerty_colab package
PROJECT_ROOT = '/content/drive/MyDrive/Brain2qwerty'   # <-- adjust
assert os.path.isdir(os.path.join(PROJECT_ROOT, 'brain2qwerty_colab')), 'package not found'
sys.path.insert(0, PROJECT_ROOT)

# Option B (alternative): clone your own fork instead
# !git clone https://github.com/<you>/brain2qwerty.git /content/brain2qwerty
# PROJECT_ROOT = '/content/brain2qwerty'; sys.path.insert(0, PROJECT_ROOT)

In [ ]:
# 4. Paths + Hugging Face auth (the SpanishBCBL dataset is gated)
import os
os.environ['BRAIN2QWERTY_STUDIES'] = '/content/drive/MyDrive/brain2qwerty_data/studies'  # raw data lives on Drive (survives restarts)
os.environ['BRAIN2QWERTY_CACHE']   = '/content/cache'    # feature cache on the fast local disk
os.environ['BRAIN2QWERTY_RESULTS'] = '/content/drive/MyDrive/brain2qwerty_runs/combined'  # checkpoints on Drive

from huggingface_hub import login
login()  # paste your HF token (or set HF_TOKEN before this cell)

In [ ]:
# 4b. OPTIONAL: use a pre-warmed cache from your cluster instead of reprocessing
# 1) On the cluster:  python -m brain2qwerty_colab.main cache --subjects S1 S2
# 2) tar the $BRAIN2QWERTY_CACHE folder and upload it next to this project on Drive
# 3) Point BRAIN2QWERTY_CACHE at the extracted copy on the local VM disk:
import tarfile, os
ARCHIVE = '/content/drive/MyDrive/b2q_cache.tar.gz'   # <-- adjust / set None to skip
if ARCHIVE and os.path.exists(ARCHIVE):
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall('/content')
    os.environ['BRAIN2QWERTY_CACHE'] = '/content/b2q_cache'  # folder name inside the tar
    print('using pre-warmed cache:', os.environ['BRAIN2QWERTY_CACHE'])
# NOTE: cache keys are config-based — train with the SAME subjects/code version
# used to warm the cache, or lookups will miss and recompute (needs raw data).

In [ ]:
# 5. Choose subjects and preset
SUBJECTS = ['S1', 'S2', 'S3']          # e.g. a 3-subject subset; None = all 19 participants
TIMELINE_QUERY = None                   # e.g. "subject in ['S1','S2','S3']" to skip loading other recordings
SMALL_ENCODER = False                   # True = 512-dim encoder for free-tier GPUs / fast iteration

from brain2qwerty_colab import studies  # registers the Pinet2024Meg study
from brain2qwerty_colab.config.xp_config import colab_config, debug_config

cfg = colab_config(subjects=SUBJECTS, timeline_query=TIMELINE_QUERY, small=SMALL_ENCODER)
print('subjects:', SUBJECTS or 'all 19')
print('epochs:', cfg['max_epochs'], '| staged: CTC@0, +contrastive@%d, +LLM@%d'
      % (cfg['contrastive_start_epoch'], cfg['llm_start_epoch']))

In [ ]:
# 6. (recommended) pre-warm the feature cache — downloads the dataset and
# extracts MEG features once; later runs reuse the cache.
# Start with debug_config() for a 1-timeline sanity check of the whole data path:
from brain2qwerty_colab.main import Experiment
Experiment(**debug_config(subjects=SUBJECTS)).data.build()
print('debug data path OK')

In [ ]:
# 7. Train (single GPU, Colab preset).
# Alternatively from the shell:
#   !python -m brain2qwerty_colab.main colab --subjects S1 S2 S3
#   !python -m brain2qwerty_colab.main colab --small --subjects S1
xp = Experiment(**cfg)
xp.run()

In [ ]:
# 8. Evaluate the best checkpoint on the test split (writes predictions_test.csv)
import os
ckpt = os.path.join(os.environ['BRAIN2QWERTY_RESULTS'], 'best_llm.ckpt')
eval_cfg = dict(cfg)
eval_cfg['eval_only'] = True
eval_cfg['ckpt_path'] = ckpt
Experiment(**eval_cfg).run()

In [ ]:
# 9. Per-subject metrics from the predictions CSV
from brain2qwerty_colab.scripts.extract_predictions import main as summarize
summarize(['--input', os.environ['BRAIN2QWERTY_RESULTS'], '--split', 'test'])

## Notes

- **Subject selection**: `subjects=[...]` filters participants *after* the V1 merge rules
  (so `S18` counts as `S1`, etc.) and *before* integer factorisation. Combine with
  `timeline_query="subject in [...]"` to also skip downloading/processing other recordings
  (note: the query sees raw subject ids like `S1`, `S18` before merging).
- **Staged schedule** (Colab preset): CTC from epoch 0, +word-contrastive at 100, +LLM at 150,
  200 epochs total. `best_ctc.ckpt` tracks the encoder (val CER), `best_llm.ckpt` the full pipeline (val WER).
- **Checkpoints and the raw dataset are on Drive**, the feature cache is on the local VM disk
  (rebuilt if the VM resets; re-run cell 6 first).
- Free-tier T4: use `SMALL_ENCODER = True`, keep `SUBJECTS` to 1–3 participants.